# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# PWE-EFDのスペクトルを見てみる

In [ ]:
import pyspedas as psp
import pytplot as pt
import ergpyspedas.erg as ergpy

pt.del_data('*')

time_range = ['20191016/00:00:00', '20191101/00:00:00']

ergpy.pwe_efd(trange=time_range, level='l2', datatype='spec', get_support_data=True)

In [ ]:
efd_spectra         = psp.get_data('erg_pwe_efd_l2_spec_spectra', xarray=True).sortby('time')
efd_spectra_flag    = psp.get_data('erg_pwe_efd_l2_spec_quality_flag', xarray=True).sortby('time')

bad_times = efd_spectra_flag.time.where(efd_spectra_flag != 0, drop=True)

print(efd_spectra)
print('')
print(efd_spectra_flag)
print('')
print(bad_times)

In [ ]:
import numpy as np

qf_on_spec = efd_spectra_flag.reindex(
    time=efd_spectra.time,
    method="nearest",
    tolerance=np.timedelta64(500, "ms")
)

efd_spectra_qf = efd_spectra.where(qf_on_spec == 0, np.nan)

# LEP-i, LEP-eのデータ存在時間の確認

In [ ]:
ergpy.lepe(trange=time_range, datatype='omniflux', level='l2')
ergpy.lepi(trange=time_range, datatype='omniflux', level='l2')

In [ ]:
LEPe_FEDO_omniflux = psp.get_data('erg_lepe_l2_omniflux_FEDO', xarray=True).sortby('time')
LEPi_FPDO_omniflux = psp.get_data('erg_lepi_l2_omniflux_FPDO', xarray=True).sortby('time')

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

def find_time_gaps_with_bad_omniflux(
    flux_da,
    threshold=pd.Timedelta(minutes=1),
    min_bad_duration=pd.Timedelta(seconds=30),
    invalid_fraction_threshold=0.3,
    time_dim="time",
    include_nonfinite=True,
    include_negative=True,
):
    """
    time gap に加えて、
    omniflux の使用不可データ割合が invalid_fraction_threshold 以上の状態が
    min_bad_duration 以上継続した場合に bad time gap として抽出する。

    Parameters
    ----------
    flux_da : xr.DataArray
        例: erg_lepe_l2_omniflux_FEDO
        dims は通常 (time, energy) を想定。
    threshold : pd.Timedelta
        隣接時刻差がこの値以上なら通常の time gap とみなす。
    min_bad_duration : pd.Timedelta
        bad 状態がこの時間以上継続した場合のみ gap として採用する。
    invalid_fraction_threshold : float
        各時刻で使用不可データがこの割合以上なら bad sample とする。
        例: 0.3 なら 30%以上。
    time_dim : str
        時間次元名。
    include_nonfinite : bool
        True の場合、NaN/inf を使用不可とみなす。
    include_negative : bool
        True の場合、負値を使用不可とみなす。

    Returns
    -------
    gap_ranges_df : pd.DataFrame
    """

    da = flux_da.sortby(time_dim)
    times = pd.DatetimeIndex(pd.to_datetime(da[time_dim].values))

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "reason",
        "bad_start_time",
        "bad_end_time",
        "bad_duration_sec",
        "bad_duration_min",
        "n_bad_time_samples",
        "n_invalid_values",
        "invalid_fraction_max",
        "invalid_fraction_mean",
    ]

    if len(times) == 0:
        return pd.DataFrame(columns=columns)

    rows = []

    # ============================================================
    # 1. 通常の time gap
    # ============================================================
    if len(times) >= 2:
        dt = times[1:] - times[:-1]
        gap_mask = dt >= threshold

        for i in np.where(gap_mask)[0]:
            rows.append({
                "gap_start_time": times[i],
                "gap_end_time": times[i + 1],
                "gap_duration_sec": dt[i].total_seconds(),
                "gap_duration_min": dt[i].total_seconds() / 60,
                "reason": "time_gap",
                "bad_start_time": pd.NaT,
                "bad_end_time": pd.NaT,
                "bad_duration_sec": np.nan,
                "bad_duration_min": np.nan,
                "n_bad_time_samples": 0,
                "n_invalid_values": 0,
                "invalid_fraction_max": np.nan,
                "invalid_fraction_mean": np.nan,
            })

    # ============================================================
    # 2. 使用不可データ割合を評価
    # ============================================================
    reduce_dims = [d for d in da.dims if d != time_dim]

    invalid_da = xr.zeros_like(da, dtype=bool)

    if include_nonfinite:
        invalid_da = invalid_da | (~np.isfinite(da))

    if include_negative:
        invalid_da = invalid_da | (da < 0)

    if len(reduce_dims) > 0:
        n_invalid_each_time = invalid_da.sum(dim=reduce_dims)
        n_total_each_time = xr.ones_like(invalid_da, dtype=int).sum(dim=reduce_dims)

        invalid_fraction = n_invalid_each_time / n_total_each_time
        bad_time = invalid_fraction >= invalid_fraction_threshold

        n_invalid_each_time = n_invalid_each_time.values.astype(int)
        invalid_fraction_each_time = invalid_fraction.values.astype(float)
        bad_time = bad_time.values.astype(bool)

    else:
        n_invalid_each_time = invalid_da.values.astype(int)
        invalid_fraction_each_time = invalid_da.values.astype(float)
        bad_time = invalid_fraction_each_time >= invalid_fraction_threshold

    bad_indices = np.where(bad_time)[0]

    if len(bad_indices) > 0:
        # 連続する bad sample を block 化
        split_points = np.where(np.diff(bad_indices) > 1)[0]

        block_starts = np.r_[bad_indices[0], bad_indices[split_points + 1]]
        block_ends   = np.r_[bad_indices[split_points], bad_indices[-1]]

        for i0, i1 in zip(block_starts, block_ends):

            bad_start = times[i0]

            # bad interval の終端は最後の bad sample の次時刻とする
            if i1 < len(times) - 1:
                bad_interval_end = times[i1 + 1]
            else:
                if len(times) >= 2:
                    cadence = pd.Series(times[1:] - times[:-1]).median()
                else:
                    cadence = pd.Timedelta(0)

                bad_interval_end = times[i1] + cadence

            bad_duration = bad_interval_end - bad_start

            # 継続時間が足りなければ採用しない
            if bad_duration < min_bad_duration:
                continue

            # gap としては直前の good sample から直後の good sample まで
            if i0 > 0:
                gap_start = times[i0 - 1]
            else:
                gap_start = times[i0]

            if i1 < len(times) - 1:
                gap_end = times[i1 + 1]
            else:
                gap_end = bad_interval_end

            gap_dt = gap_end - gap_start

            invalid_frac_block = invalid_fraction_each_time[i0:i1 + 1]

            rows.append({
                "gap_start_time": gap_start,
                "gap_end_time": gap_end,
                "gap_duration_sec": gap_dt.total_seconds(),
                "gap_duration_min": gap_dt.total_seconds() / 60,
                "reason": "bad_omniflux_fraction",
                "bad_start_time": bad_start,
                "bad_end_time": times[i1],
                "bad_duration_sec": bad_duration.total_seconds(),
                "bad_duration_min": bad_duration.total_seconds() / 60,
                "n_bad_time_samples": int(i1 - i0 + 1),
                "n_invalid_values": int(n_invalid_each_time[i0:i1 + 1].sum()),
                "invalid_fraction_max": float(np.nanmax(invalid_frac_block)),
                "invalid_fraction_mean": float(np.nanmean(invalid_frac_block)),
            })

    gap_ranges_df = pd.DataFrame(rows, columns=columns)

    if len(gap_ranges_df) == 0:
        return gap_ranges_df

    gap_ranges_df = gap_ranges_df.sort_values(
        ["gap_start_time", "gap_end_time"]
    ).reset_index(drop=True)

    return gap_ranges_df

In [ ]:
LEPe_omniflux_gap_ranges_df = find_time_gaps_with_bad_omniflux(
    LEPe_FEDO_omniflux,
    threshold=pd.Timedelta(minutes=1),
    min_bad_duration=pd.Timedelta(minutes=1),
    invalid_fraction_threshold=0.3,
)

print(f"1分以上の time gap 数: {len(LEPe_omniflux_gap_ranges_df)}")
display(LEPe_omniflux_gap_ranges_df)

In [ ]:
LEPi_omniflux_gap_ranges_df = find_time_gaps_with_bad_omniflux(
    LEPi_FPDO_omniflux,
    threshold=pd.Timedelta(minutes=1),
    min_bad_duration=pd.Timedelta(minutes=1),
    invalid_fraction_threshold=0.3,
)

print(f"1分以上の time gap 数: {len(LEPi_omniflux_gap_ranges_df)}")
display(LEPi_omniflux_gap_ranges_df)

# `erg_att_izras`と`erg_att_izdec`を取得

In [ ]:
import numpy as np
import pandas as pd
import pyspedas as psp

ergpy.satellite.erg.att.att.att(trange=time_range)

# dsi2j2000.py / erg_interpolate_att.py と同じ処理
psp.degap('erg_att_izras', dt=8., margin=0.5)
psp.degap('erg_att_izdec', dt=8., margin=0.5)

att_izras = psp.get_data('erg_att_izras', xarray=True)
att_izdec = psp.get_data('erg_att_izdec', xarray=True)

In [ ]:
def find_nan_intervals_att(att_izras, att_izdec, time_dim="time"):
    """
    erg_att_izras / erg_att_izdec のどちらかが NaN になる連続区間を抽出する。
    gap_start_time / gap_end_time は、NaN block を挟む直前・直後の正常 grid。
    """

    izras = att_izras.sortby(time_dim)
    izdec = att_izdec.sortby(time_dim)

    times = pd.DatetimeIndex(pd.to_datetime(izras[time_dim].values))

    ras = np.asarray(izras.values)
    dec = np.asarray(izdec.values)

    bad = np.isnan(ras) | np.isnan(dec)
    bad_indices = np.where(bad)[0]

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "bad_start_time",
        "bad_end_time",
        "bad_duration_sec",
        "bad_duration_min",
        "n_bad_time_samples",
        "n_nan_izras",
        "n_nan_izdec",
        "reason",
    ]

    if len(bad_indices) == 0:
        return pd.DataFrame(columns=columns)

    rows = []

    split_points = np.where(np.diff(bad_indices) > 1)[0]
    block_starts = np.r_[bad_indices[0], bad_indices[split_points + 1]]
    block_ends = np.r_[bad_indices[split_points], bad_indices[-1]]

    cadence = pd.Series(times[1:] - times[:-1]).median() if len(times) >= 2 else pd.Timedelta(0)

    for i0, i1 in zip(block_starts, block_ends):
        bad_start = times[i0]

        if i1 < len(times) - 1:
            bad_interval_end = times[i1 + 1]
        else:
            bad_interval_end = times[i1] + cadence

        bad_duration = bad_interval_end - bad_start

        # NaN block を挟む正常 grid の間を gap とする
        gap_start = times[i0 - 1] if i0 > 0 else times[i0]
        gap_end = times[i1 + 1] if i1 < len(times) - 1 else bad_interval_end
        gap_duration = gap_end - gap_start

        rows.append({
            "gap_start_time": gap_start,
            "gap_end_time": gap_end,
            "gap_duration_sec": gap_duration.total_seconds(),
            "gap_duration_min": gap_duration.total_seconds() / 60,
            "bad_start_time": bad_start,
            "bad_end_time": times[i1],
            "bad_duration_sec": bad_duration.total_seconds(),
            "bad_duration_min": bad_duration.total_seconds() / 60,
            "n_bad_time_samples": int(i1 - i0 + 1),
            "n_nan_izras": int(np.isnan(ras[i0:i1 + 1]).sum()),
            "n_nan_izdec": int(np.isnan(dec[i0:i1 + 1]).sum()),
            "reason": "att_izras_or_izdec_nan",
        })

    return pd.DataFrame(rows, columns=columns)

In [ ]:
att_nan_intervals_df = find_nan_intervals_att(att_izras, att_izdec)

print(f"time gap 数: {len(att_nan_intervals_df)}")
display(att_nan_intervals_df)

# 第一段階検証 Especについて

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# gap range を time mask に変換する関数
# ============================================================

def make_not_in_gap_mask(times, gap_ranges_df_list):
    """
    times が、複数の gap_ranges_df のどの時間範囲にも含まれない mask を返す。

    gap_ranges_df は以下のどちらかの列を持つ想定:
      - missing_start_time, missing_end_time
      - gap_start_time, gap_end_time
    """

    times_pd = pd.DatetimeIndex(pd.to_datetime(times))

    in_gap = np.zeros(len(times_pd), dtype=bool)

    for gap_df in gap_ranges_df_list:
        if gap_df is None or len(gap_df) == 0:
            continue

        if {"missing_start_time", "missing_end_time"}.issubset(gap_df.columns):
            start_col = "missing_start_time"
            end_col   = "missing_end_time"
        elif {"gap_start_time", "gap_end_time"}.issubset(gap_df.columns):
            start_col = "gap_start_time"
            end_col   = "gap_end_time"
        else:
            raise ValueError(
                "gap_ranges_df must have either "
                "['missing_start_time', 'missing_end_time'] or "
                "['gap_start_time', 'gap_end_time'] columns."
            )

        for _, row in gap_df.iterrows():
            start = pd.to_datetime(row[start_col])
            end   = pd.to_datetime(row[end_col])

            if pd.isna(start) or pd.isna(end):
                continue

            in_gap |= (times_pd >= start) & (times_pd <= end)

    not_in_gap = ~in_gap

    return not_in_gap

In [ ]:
# spec_bins = 3..32 の範囲で、各 time ごとに 60% 以上のデータが 1e-4 を超える time を抽出
subset = efd_spectra_qf.sel(spec_bins=slice(3, 32))

valid = subset.notnull()
above = valid & (subset > 1e-4)

frac_above = above.sum(dim="v_dim") / valid.sum(dim="v_dim")
mask = frac_above >= 0.6

# ============================================================
# 3点以上連続している区間を丸ごと残す
# ============================================================

mask_np = mask.values.astype(bool)

kernel = np.ones(3, dtype=int)
count3 = np.convolve(mask_np.astype(int), kernel, mode="valid")
window3 = count3 == 3

mask_consecutive_3_np = np.zeros_like(mask_np, dtype=bool)

for i, ok in enumerate(window3):
    if ok:
        mask_consecutive_3_np[i:i+3] = True

# ============================================================
# LEPe / LEPi gap 範囲に含まれない条件を追加
# ============================================================

not_in_lep_gap_mask = make_not_in_gap_mask(
    frac_above.time.values,
    [
        LEPe_omniflux_gap_ranges_df,
        LEPi_omniflux_gap_ranges_df,
        att_nan_intervals_df,
    ],
)

# 最終 mask
final_mask_np = mask_consecutive_3_np & not_in_lep_gap_mask

selected_times = frac_above.time.values[final_mask_np]
selected_fracs = frac_above.values[final_mask_np]

print(f"3点以上連続条件を満たす time の数: {mask_consecutive_3_np.sum()}")
print(f"LEPe/LEPi gap 除外後の time の数: {len(selected_times)}")

In [ ]:
import numpy as np
import pandas as pd


def gap_df_to_intervals_fast(gap_df):
    """
    LEPe_gap_ranges_df / LEPi_gap_ranges_df を interval DataFrame に変換する。
    """

    if gap_df is None or len(gap_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    if {"missing_start_time", "missing_end_time"}.issubset(gap_df.columns):
        start_col = "missing_start_time"
        end_col = "missing_end_time"
    elif {"gap_start_time", "gap_end_time"}.issubset(gap_df.columns):
        start_col = "gap_start_time"
        end_col = "gap_end_time"
    else:
        raise ValueError(
            "gap_df must have either "
            "['missing_start_time', 'missing_end_time'] or "
            "['gap_start_time', 'gap_end_time'] columns."
        )

    out = pd.DataFrame({
        "start_time": pd.to_datetime(gap_df[start_col]),
        "end_time": pd.to_datetime(gap_df[end_col]),
    })

    out = out.dropna()
    out = out[out["end_time"] >= out["start_time"]]

    return out.reset_index(drop=True)


def merge_intervals_fast(intervals_df):
    """
    start_time, end_time を持つ interval DataFrame を高速に merge する。
    """

    if intervals_df is None or len(intervals_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    df = intervals_df.dropna().copy()
    df = df.sort_values("start_time").reset_index(drop=True)

    starts = pd.DatetimeIndex(df["start_time"]).to_numpy()
    ends = pd.DatetimeIndex(df["end_time"]).to_numpy()

    order = np.argsort(starts)
    starts = starts[order]
    ends = ends[order]

    # ここは datetime64[ns] のまま np.maximum.accumulate できる
    cum_ends = np.maximum.accumulate(ends)

    # 新しい interval が始まる条件
    # start > 直前までの cumulative end
    new_group = np.empty(len(starts), dtype=bool)
    new_group[0] = True
    new_group[1:] = starts[1:] > cum_ends[:-1]

    group_id = np.cumsum(new_group) - 1

    merged_starts = []
    merged_ends = []

    for gid in np.unique(group_id):
        use = group_id == gid
        merged_starts.append(starts[use][0])
        merged_ends.append(ends[use].max())

    return pd.DataFrame({
        "start_time": pd.to_datetime(merged_starts),
        "end_time": pd.to_datetime(merged_ends),
    })


def make_forbidden_intervals_fast(
    bad_times=None,
    LEPe_gap_ranges_df=None,
    LEPi_gap_ranges_df=None,
    att_gap_ranges_df=None,
):
    """
    bad_times, LEPe gap, LEPi gap をまとめて禁止 interval にする。

    bad_times は point-like interval [t, t] として扱う。
    """

    dfs = []

    if bad_times is not None:
        bad_times_pd = pd.DatetimeIndex(pd.to_datetime(bad_times.values))
        bad_times_pd = bad_times_pd.dropna().sort_values().unique()
        bad_times_pd = pd.DatetimeIndex(bad_times_pd)

        if len(bad_times_pd) > 0:
            dfs.append(pd.DataFrame({
                "start_time": bad_times_pd,
                "end_time": bad_times_pd,
            }))

    df_lepe = gap_df_to_intervals_fast(LEPe_gap_ranges_df)
    df_lepi = gap_df_to_intervals_fast(LEPi_gap_ranges_df)
    df_att  = gap_df_to_intervals_fast(att_gap_ranges_df)

    if len(df_lepe) > 0:
        dfs.append(df_lepe)

    if len(df_lepi) > 0:
        dfs.append(df_lepi)

    if len(df_att) > 0:
        dfs.append(df_att)

    if len(dfs) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    intervals_df = pd.concat(dfs, ignore_index=True)
    intervals_df = merge_intervals_fast(intervals_df)

    return intervals_df

In [ ]:
def make_trimmed_time_ranges_fast(
    selected_times,
    forbidden_intervals_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
):
    """
    selected_times ± half_width の区間を作る。
    ただし forbidden intervals を含まないように、
    各 selected_time の前後で最も近い forbidden interval まで切り詰める。

    Returns
    -------
    time_ranges : list[tuple[pd.Timestamp, pd.Timestamp]]
    """

    selected_times_pd = pd.DatetimeIndex(
        pd.to_datetime(selected_times)
    ).sort_values()

    if len(selected_times_pd) == 0:
        return []

    start0 = selected_times_pd - half_width
    end0 = selected_times_pd + half_width

    if forbidden_intervals_df is None or len(forbidden_intervals_df) == 0:
        return list(zip(start0, end0))

    forbidden_intervals_df = forbidden_intervals_df.sort_values("start_time").reset_index(drop=True)

    f_starts = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["start_time"]))
    f_ends = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["end_time"]))

    # numpy datetime64[ns] として扱う
    t_np = selected_times_pd.to_numpy()
    start_np = start0.to_numpy()
    end_np = end0.to_numpy()

    f_starts_np = f_starts.to_numpy()
    f_ends_np = f_ends.to_numpy()

    # 各 t に対して、t 以下で始まる最後の forbidden interval
    i_prev = np.searchsorted(f_starts_np, t_np, side="right") - 1

    valid_prev = i_prev >= 0

    inside_forbidden = np.zeros(len(t_np), dtype=bool)

    # t が forbidden interval 内に入っているか
    inside_forbidden[valid_prev] = (
        t_np[valid_prev] <= f_ends_np[i_prev[valid_prev]]
    )

    # start 側の切り詰め
    start_trim = start_np.copy()

    use_prev_trim = np.zeros(len(t_np), dtype=bool)
    use_prev_trim[valid_prev] = (
        f_ends_np[i_prev[valid_prev]] > start_np[valid_prev]
    )

    start_trim[use_prev_trim] = (
        f_ends_np[i_prev[use_prev_trim]] + np.timedelta64(eps.value, "ns")
    )

    # next forbidden interval
    # inside でない場合、直後の interval は i_prev + 1
    i_next = i_prev + 1
    valid_next = i_next < len(f_starts_np)

    end_trim = end_np.copy()

    use_next_trim = np.zeros(len(t_np), dtype=bool)
    use_next_trim[valid_next] = (
        f_starts_np[i_next[valid_next]] < end_np[valid_next]
    )

    end_trim[use_next_trim] = (
        f_starts_np[i_next[use_next_trim]] - np.timedelta64(eps.value, "ns")
    )

    valid = (~inside_forbidden) & (end_trim > start_trim)

    start_out = pd.to_datetime(start_trim[valid])
    end_out = pd.to_datetime(end_trim[valid])

    time_ranges = list(zip(start_out, end_out))

    return time_ranges

In [ ]:
def merge_time_ranges(
    time_ranges,
    max_duration=None,
):
    """
    overlapping time ranges を merge する。
    max_duration を指定した場合、merge 後の duration がそれを超える merge はしない。
    """

    if len(time_ranges) == 0:
        return []

    time_ranges = sorted(time_ranges)

    merged = []

    for start, end in time_ranges:
        if not merged:
            merged.append((start, end))
            continue

        prev_start, prev_end = merged[-1]

        is_overlapping = start <= prev_end

        candidate_start = prev_start
        candidate_end = max(prev_end, end)

        if max_duration is None:
            ok_duration = True
        else:
            ok_duration = (candidate_end - candidate_start) <= max_duration

        if is_overlapping and ok_duration:
            merged[-1] = (candidate_start, candidate_end)
        else:
            merged.append((start, end))

    return merged

In [ ]:
import time

t0_clock = time.perf_counter()

# ============================================================
# forbidden intervals を作る
# ============================================================

forbidden_intervals_df = make_forbidden_intervals_fast(
    bad_times=bad_times,
    LEPe_gap_ranges_df=LEPe_omniflux_gap_ranges_df,
    LEPi_gap_ranges_df=LEPi_omniflux_gap_ranges_df,
    att_gap_ranges_df=att_nan_intervals_df
)

print(f"forbidden intervals: {len(forbidden_intervals_df)}")

# ============================================================
# selected_times ±15 min を forbidden interval で切る
# ============================================================

time_ranges = make_trimmed_time_ranges_fast(
    selected_times=selected_times,
    forbidden_intervals_df=forbidden_intervals_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
)

print(f"time_ranges before merge: {len(time_ranges)}")

# ============================================================
# merge
# ============================================================

merged_ranges = merge_time_ranges(
    time_ranges,
    max_duration=None,
    # max_duration=pd.Timedelta(minutes=60),  # 必要ならこちら
)

# ============================================================
# DataFrame 化
# ============================================================

merged_ranges_df = pd.DataFrame(
    merged_ranges,
    columns=["start_time", "end_time"]
)

if len(merged_ranges_df) > 0:
    merged_ranges_df["duration_minutes"] = (
        merged_ranges_df["end_time"] - merged_ranges_df["start_time"]
    ).dt.total_seconds() / 60

    merged_ranges_df = merged_ranges_df[
        merged_ranges_df["duration_minutes"] > 30
    ].reset_index(drop=True)
else:
    merged_ranges_df["duration_minutes"] = []

elapsed = time.perf_counter() - t0_clock

print(f"元の選択時間数: {len(selected_times)}")
print(f"30分超のマージ後時間範囲数: {len(merged_ranges_df)}")
print(f"elapsed: {elapsed:.2f} sec")
print("\n有効なマージ時間範囲:")
print(merged_ranges_df)

# MGF dataのFFT

In [ ]:
ergpy.mgf(trange=time_range, level='l2', datatype='64hz', get_support_data=True, no_update=True)

In [ ]:
import xarray as xr

B64_data_dsi = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time')
B64_data_dsi_quality_flag = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time')

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

In [ ]:
B64_data_dsi_qf_total = np.sqrt(xr.dot(B64_data_dsi_qf, B64_data_dsi_qf, dim='v_dim'))

In [ ]:
m_e     = 9.1093837E-31    #[kg]
m_H     = 1.6726219e-27  # kg
m_He    = m_H * 4.
m_O     = m_H * 16.
elementary_charge = 1.60217663E-19  #[A s]

f_cH    = elementary_charge * B64_data_dsi_qf_total*1E-9 / m_H  / 2. / np.pi
f_cHe   = elementary_charge * B64_data_dsi_qf_total*1E-9 / m_He / 2. / np.pi
f_cO    = elementary_charge * B64_data_dsi_qf_total*1E-9 / m_O / 2. / np.pi

In [ ]:
#from pathlib import Path
#import numpy as np
#import pandas as pd
#from matplotlib.colors import Normalize
#import matplotlib.pyplot as plt
#import matplotlib.dates as mdates
#
#path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
#path_base_save_plot = Path(path_base_save_plot + f'/PWE-EFD_spec')
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#def edges_numeric(arr):
#    arr = np.asarray(arr, dtype=float)
#    if arr.size < 2:
#        return np.concatenate((arr, arr + 1.0))
#    d = np.diff(arr) / 2.0
#    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))
#
#vmax = np.log10(1E1)
#vmin = np.log10(1e-4)
#
#for idx, row in merged_ranges_df.iterrows():
#    start = row["start_time"]
#    end = row["end_time"]
#
#    if pd.isna(start) or pd.isna(end):
#        continue
#
#    subset_range = efd_spectra_qf.sel(time=slice(start, end))
#
#    if subset_range.sizes.get("time", 0) < 2:
#        continue
#
#    times = pd.to_datetime(subset_range.time.values)
#
#    if "spec_bins" in subset_range.coords:
#        freqs = subset_range["spec_bins"].values
#    elif "v" in subset_range.coords:
#        freqs = subset_range["v"].values
#    else:
#        freqs = np.arange(subset_range.sizes["v_dim"])
#
#    data_range = subset_range.values
#    data_range = np.where(
#        np.isfinite(data_range) & (data_range > 0),
#        data_range,
#        np.nan
#    )
#
#    logdata_range = np.log10(data_range)
#
#    if np.isnan(logdata_range).all():
#        continue
#
#    t_num = mdates.date2num(times.to_pydatetime())
#    t_edges = edges_numeric(t_num)
#    f_edges = edges_numeric(freqs)
#
#    fig_i, ax_i = plt.subplots(figsize=(12, 4.5))
#
#    pcm_i = ax_i.pcolormesh(
#        t_edges,
#        f_edges,
#        logdata_range.T,
#        shading="auto",
#        cmap="turbo",
#        norm=Normalize(vmin=vmin, vmax=vmax),
#    )
#
#    f_cH_plot = f_cH.sel(time=slice(start, end))
#    f_cHe_plot = f_cHe.sel(time=slice(start, end))
#    f_cO_plot = f_cO.sel(time=slice(start, end))
#
#    ax_i.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
#    ax_i.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
#    ax_i.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')
#
#    ax_i.set_xlabel("Time")
#    ax_i.xaxis_date()
#    ax_i.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
#    ax_i.set_ylabel("Frequency [Hz]")
#    ax_i.set_title(
#        f"PWE-EFD spec ({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
#    )
#    ax_i.set_yscale("log")
#    ax_i.set_ylim(3, 32)
#    ax_i.set_xlim(start.to_pydatetime(), end.to_pydatetime())
#
#    fig_i.colorbar(pcm_i, ax=ax_i, label=r"log10 PSD [$\mathrm{(mV/m)}^2/\mathrm{Hz}$]")
#    plt.tight_layout()
#
#    # =========================
#    # save figure
#    # =========================
#
#    # start の月・日で階層フォルダを作る
#    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
#    save_dir.mkdir(parents=True, exist_ok=True)
#
#    filename = (
#        f"efd_spectra_qf_"
#        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
#    )
#
#    save_path = save_dir / filename
#
#    fig_i.savefig(save_path, dpi=200, bbox_inches="tight")
#    plt.close(fig_i)
#
#    print(f"saved: {save_path}")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def _estimate_sampling_rate_from_time(time_values):
    """
    datetime64 time coordinate から sampling rate を推定する。
    """
    time = pd.DatetimeIndex(pd.to_datetime(time_values))
    t_sec = (time - time[0]).total_seconds().to_numpy()

    dt_arr = np.diff(t_sec)
    dt_arr = dt_arr[np.isfinite(dt_arr) & (dt_arr > 0)]

    if len(dt_arr) == 0:
        raise ValueError("time 軸から dt を推定できない。")

    dt = np.median(dt_arr)
    fs = 1.0 / dt

    return fs, dt


def _one_sided_psd(x, fs, use_hann=True, remove_mean=True):
    """
    1D waveform から one-sided PSD を計算する。

    入力 x の単位が nT なら、出力 PSD は nT^2/Hz。
    """

    x = np.asarray(x, dtype=float)

    if np.any(~np.isfinite(x)):
        raise ValueError("x に NaN/Inf が含まれている。")

    if remove_mean:
        x = x - np.mean(x)

    N = len(x)

    if use_hann:
        window = np.hanning(N)
    else:
        window = np.ones(N)

    xw = x * window

    fft_vals = np.fft.rfft(xw)
    freq = np.fft.rfftfreq(N, d=1.0 / fs)

    psd = np.abs(fft_vals) ** 2 / (fs * np.sum(window ** 2))

    # one-sided correction
    if N % 2 == 0:
        # even N: DC と Nyquist 以外を2倍
        psd[1:-1] *= 2.0
    else:
        # odd N: DC 以外を2倍
        psd[1:] *= 2.0

    return freq, psd


def _bin_psd_to_1hz_bins(freq, psd, spec_bins=np.arange(3, 33), bin_width=1.0):
    """
    raw FFT PSD を 1 Hz ごとの spec_bins にまとめる。

    spec_bins は bin center として扱う。
    spec_bins=3 は 2.5--3.5 Hz の平均 PSD。
    spec_bins=32 は 31.5--32.5 Hz だが、Nyquist により実質 31.5--32 Hz。
    """

    freq = np.asarray(freq)
    psd = np.asarray(psd)

    out = np.full(len(spec_bins), np.nan)

    for i, fc in enumerate(spec_bins):
        f0 = fc - bin_width / 2.0
        f1 = fc + bin_width / 2.0

        if i == len(spec_bins) - 1:
            use = (freq >= f0) & (freq <= f1)
        else:
            use = (freq >= f0) & (freq < f1)

        if np.any(use):
            out[i] = np.nanmean(psd[use])

    return out

In [ ]:
def make_mgf64_spectra_like_efd(
    da_B,
    target_times=None,
    spec_bins=np.arange(3, 33),
    window_sec=1.0,
    step_sec=1.0,
    min_valid_fraction=1.0,
    gap_factor=1.5,
    use_hann=True,
    remove_mean=True,
    name="B64_spectra_qf",
):
    """
    MGF 64 Hz total magnetic field から EFD spec と同様の
    time x spec_bins スペクトルプロダクトを作る。

    Parameters
    ----------
    da_B : xr.DataArray
        time 軸を持つ 1D DataArray。
        単位は nT を想定。
        quality flag 不良点は NaN になっているものとする。
    target_times : array-like or None
        出力する spectrum の中心時刻。
        EFD と完全に同じ time grid にしたい場合は
        target_times=efd_spectra_qf.time.values
        とする。
        None の場合は step_sec ごとの時刻を自動生成する。
    spec_bins : array-like
        固定 frequency bins。デフォルトは 3--32 Hz。
    window_sec : float
        FFT 窓長。64 Hz sampling で 1秒なら df=1 Hz。
    step_sec : float
        target_times=None の場合の出力 cadence。
    min_valid_fraction : float
        各窓内で必要な有限値の割合。
        1.0 なら NaN が1点でもあれば、その窓は NaN spectrum。
    gap_factor : float
        窓内の time gap が通常 dt の gap_factor 倍を超えたら無効化。
    use_hann : bool
        Hann window を使うか。
    remove_mean : bool
        各窓で mean を除去するか。
    name : str
        出力 DataArray 名。

    Returns
    -------
    spectra : xr.DataArray
        dims: time, spec_bins
        units: nT^2/Hz
    """

    da_B = da_B.sortby("time")

    time = pd.DatetimeIndex(pd.to_datetime(da_B.time.values))
    values = da_B.values.astype(float)

    fs, dt = _estimate_sampling_rate_from_time(time)

    nperseg = int(round(window_sec * fs))

    if nperseg < 2:
        raise ValueError("window_sec が短すぎる。")

    # 1秒窓、64 Hzなら nperseg = 64
    actual_window_sec = nperseg / fs
    df = fs / nperseg
    nyq = fs / 2.0

    print(f"Estimated fs        : {fs:.6f} Hz")
    print(f"Estimated dt        : {dt:.6f} sec")
    print(f"nperseg             : {nperseg}")
    print(f"actual window length: {actual_window_sec:.6f} sec")
    print(f"df                  : {df:.6f} Hz")
    print(f"Nyquist             : {nyq:.6f} Hz")

    if np.max(spec_bins) > nyq + 1e-6:
        print(
            f"Warning: spec_bins max = {np.max(spec_bins)} Hz exceeds Nyquist = {nyq:.3f} Hz"
        )

    half = nperseg // 2

    # target_times を作る
    if target_times is None:
        t0 = time[0] + pd.Timedelta(seconds=actual_window_sec / 2.0)
        t1 = time[-1] - pd.Timedelta(seconds=actual_window_sec / 2.0)

        target_times = pd.date_range(
            start=t0,
            end=t1,
            freq=pd.to_timedelta(step_sec, unit="s"),
        )
    else:
        target_times = pd.DatetimeIndex(pd.to_datetime(target_times))

    spectra = np.full((len(target_times), len(spec_bins)), np.nan)
    valid_fraction_arr = np.full(len(target_times), np.nan)
    n_valid_arr = np.full(len(target_times), 0)
    is_valid_window = np.full(len(target_times), False)

    time_ns = time.asi8

    for it, tc in enumerate(target_times):
        tc_ns = pd.Timestamp(tc).value

        # 中心時刻に最も近い index
        ic = np.searchsorted(time_ns, tc_ns)

        # even nperseg の場合、中心の取り方は少し任意性がある。
        # ここでは ic を中心近傍として nperseg 点を取る。
        i0 = ic - half
        i1 = i0 + nperseg

        if i0 < 0 or i1 > len(values):
            continue

        x = values[i0:i1]
        t_win = time[i0:i1]

        if len(x) != nperseg:
            continue

        # time gap check
        t_win_sec = (t_win - t_win[0]).total_seconds().to_numpy()
        dt_win = np.diff(t_win_sec)

        if len(dt_win) == 0:
            continue

        if np.nanmax(dt_win) > gap_factor * dt:
            continue

        finite = np.isfinite(x)
        valid_fraction = finite.sum() / len(x)

        valid_fraction_arr[it] = valid_fraction
        n_valid_arr[it] = finite.sum()

        if valid_fraction < min_valid_fraction:
            continue

        # 今回は quality flag 不良点を補間しない方針。
        # min_valid_fraction=1.0 なら、ここに来る時点で NaN はない。
        if np.any(~finite):
            continue

        freq, psd = _one_sided_psd(
            x,
            fs=fs,
            use_hann=use_hann,
            remove_mean=remove_mean,
        )

        spectra[it, :] = _bin_psd_to_1hz_bins(
            freq=freq,
            psd=psd,
            spec_bins=spec_bins,
            bin_width=1.0,
        )

        is_valid_window[it] = True

    spectra_da = xr.DataArray(
        spectra,
        coords={
            "time": target_times,
            "spec_bins": spec_bins,
            "valid_fraction": ("time", valid_fraction_arr),
            "n_valid": ("time", n_valid_arr),
            "is_valid_window": ("time", is_valid_window),
        },
        dims=("time", "spec_bins"),
        name=name,
        attrs={
            "units": "nT^2/Hz",
            "input_units": "nT",
            "method": "sliding-window one-sided FFT PSD",
            "window": "hann" if use_hann else "boxcar",
            "remove_mean": bool(remove_mean),
            "sampling_rate_Hz": float(fs),
            "dt_sec": float(dt),
            "nperseg": int(nperseg),
            "window_sec": float(actual_window_sec),
            "df_Hz": float(df),
            "nyquist_Hz": float(nyq),
            "spec_bins": "frequency bin centers [Hz]",
            "bin_width_Hz": 1.0,
            "min_valid_fraction": float(min_valid_fraction),
            "quality_flag_handling": "NaN samples are not interpolated; windows with insufficient valid samples are set to NaN.",
        },
    )

    return spectra_da

In [ ]:
#B64_spectra_qf = make_mgf64_spectra_like_efd(
#    B64_data_dsi_qf_total,
#    target_times=efd_spectra_qf.time.values,  # EFD spec と同じ time
#    spec_bins=np.arange(3, 33),               # 3--32 Hz
#    window_sec=1.0,
#    min_valid_fraction=1.,                   # NaN が1点でもあれば無効
#    use_hann=True,
#    remove_mean=True,
#)
#
#print(B64_spectra_qf)

In [ ]:
#from pathlib import Path
#import pandas as pd
#import numpy as np
#
## 保存先
#path_save_nc = Path('/mnt/j/observation_data/statistical_analysis_arase/Arase_analysis_save_data/B64_spectra')
#path_save_nc.mkdir(parents=True, exist_ok=True)
#
## ファイル名を time range から作る
#t0 = pd.to_datetime(time_range[0])
#t1 = pd.to_datetime(time_range[-1])
#
#filename = (
#    f"B64_spectra_qf_"
#    f"{t0:%Y%m%d_%H%M%S}_to_{t1:%Y%m%d_%H%M%S}.nc"
#)
#
#save_path = path_save_nc / filename
#
#B64_spectra_qf.attrs.update({
#    "long_name": "MGF 64 Hz total magnetic field spectrum",
#    "units": "nT^2/Hz",
#    "input_data": "B64_data_dsi_qf_total",
#    "quality_flag": "Bad-quality samples were set to NaN before FFT.",
#    "frequency_bins": "1 Hz bins from 3 to 32 Hz",
#})
#
## DataArray -> Dataset
#ds_B64_spectra_qf = B64_spectra_qf.to_dataset(name="B64_spectra_qf")
#
## 念のため、NetCDFで扱いやすい属性に整える
#for key, val in list(ds_B64_spectra_qf.attrs.items()):
#    if isinstance(val, (bool, np.bool_)):
#        ds_B64_spectra_qf.attrs[key] = str(val)
#
#for var in ds_B64_spectra_qf.variables:
#    for key, val in list(ds_B64_spectra_qf[var].attrs.items()):
#        if isinstance(val, (bool, np.bool_)):
#            ds_B64_spectra_qf[var].attrs[key] = str(val)
#
## 保存
#ds_B64_spectra_qf.to_netcdf(save_path)
#
#print(f"saved: {save_path}")

In [ ]:
from pathlib import Path
import pandas as pd
import xarray as xr

t0 = pd.to_datetime(time_range[0])
t1 = pd.to_datetime(time_range[-1])

filename = (
    f"B64_spectra_qf_"
    f"{t0:%Y%m%d_%H%M%S}_to_{t1:%Y%m%d_%H%M%S}.nc"
)

data_path = Path('/mnt/j/observation_data/statistical_analysis_arase/Arase_analysis_save_data/B64_spectra/' + filename)
ds_loaded = xr.open_dataset(data_path)
B64_spectra_qf = ds_loaded["B64_spectra_qf"]

print(B64_spectra_qf)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize
from pathlib import Path


def edges_numeric(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return np.concatenate((arr, arr + 1.0))
    d = np.diff(arr) / 2.0
    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))


# ============================================================
# save base path
# ============================================================

path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
path_base_save_plot = Path(path_base_save_plot + f'/MGF_spec')
os.makedirs(path_base_save_plot, exist_ok=True)



# ============================================================
# plot settings
# ============================================================

vmin = -4
vmax = -1

for idx, row in merged_ranges_df.iterrows():

    start = pd.to_datetime(row["start_time"])
    end   = pd.to_datetime(row["end_time"])

    if pd.isna(start) or pd.isna(end):
        continue

    f_cH_plot = f_cH.sel(time=slice(start, end))
    f_cHe_plot = f_cHe.sel(time=slice(start, end))
    f_cO_plot = f_cO.sel(time=slice(start, end))

    # xarray sel 用
    B64_spectra_qf_analysis = B64_spectra_qf.sel(
        time=slice(start, end)
    )

    # time が少なすぎる場合は skip
    if B64_spectra_qf_analysis.sizes.get("time", 0) < 2:
        print(f"skip: too few time points: {start} - {end}")
        continue

    data = B64_spectra_qf_analysis.values
    data = np.where(np.isfinite(data) & (data > 0), data, np.nan)

    if np.isnan(data).all():
        print(f"skip: all NaN: {start} - {end}")
        continue

    logdata = np.log10(data)

    times = pd.DatetimeIndex(pd.to_datetime(B64_spectra_qf_analysis.time.values))
    t_num = mdates.date2num(times.to_pydatetime())
    t_edges = edges_numeric(t_num)

    freqs = B64_spectra_qf_analysis.spec_bins.values
    f_edges = edges_numeric(freqs)

    fig, ax = plt.subplots(figsize=(12, 4.5))

    pcm = ax.pcolormesh(
        t_edges,
        f_edges,
        logdata.T,
        shading="auto",
        cmap="turbo",
        norm=Normalize(vmin=vmin, vmax=vmax),
    )

    ax.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
    ax.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
    ax.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    ax.set_yscale("log")
    ax.set_ylim(3, 32)

    ax.set_xlim(start.to_pydatetime(), end.to_pydatetime())

    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency [Hz]")
    ax.set_title(
        f"MGF 64hz total spec "
        f"({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
    )

    fig.colorbar(
        pcm,
        ax=ax,
        label=r"log10 PSD [$\mathrm{nT}^2/\mathrm{Hz}$]"
    )

    plt.tight_layout()

    # ========================================================
    # save
    # ========================================================

    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"MGF_B64_spectra_"
        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
    )

    save_path = save_dir / filename

    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"saved: {save_path}")

# 各selected_timeでのMGF PSDの勾配を確認

In [ ]:
B64_spectra_qf_selected = B64_spectra_qf.sel(time=selected_times[5000])
B64_spectra_qf_selected

In [ ]:
import numpy as np
import pandas as pd


def quantify_decreasing_psd(
    psd_da,
    freq_dim="spec_bins",
    fmin=3,
    fmax=32,
):
    """
    1時刻の PSD spectrum が概ね単調減少かを定量化する。

    Parameters
    ----------
    psd_da : xr.DataArray
        dims に spec_bins を持つ 1D DataArray。
        例: B64_spectra_qf.sel(time=selected_times[5000])
    freq_dim : str
        周波数次元名。
    fmin, fmax : float
        評価する周波数範囲。

    Returns
    -------
    metrics : pd.Series
    """

    da = psd_da.sel({freq_dim: slice(fmin, fmax)})

    f = da[freq_dim].values.astype(float)
    P = da.values.astype(float)

    use = np.isfinite(f) & np.isfinite(P) & (P > 0)

    f = f[use]
    P = P[use]

    if len(f) < 3:
        return pd.Series({
            "n_points": len(f),
            "alpha": np.nan,
            "intercept": np.nan,
            "r2_loglog": np.nan,
            "rmse_log10": np.nan,
            "spearman_rho": np.nan,
            "adjacent_decrease_fraction": np.nan,
            "positive_local_slope_fraction": np.nan,
            "median_local_alpha": np.nan,
            "log10_drop_low_to_high": np.nan,
            "monotonic_decrease_score": np.nan,
        })

    # 周波数順に並べる
    order = np.argsort(f)
    f = f[order]
    P = P[order]

    logf = np.log10(f)
    logP = np.log10(P)

    # ========================================================
    # 1. log-log power-law fit
    # logP = intercept + slope * logf
    # alpha = -slope
    # ========================================================

    slope, intercept = np.polyfit(logf, logP, 1)
    alpha = -slope

    fit = intercept + slope * logf
    resid = logP - fit

    ss_res = np.sum(resid**2)
    ss_tot = np.sum((logP - np.mean(logP))**2)

    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse_log10 = np.sqrt(np.mean(resid**2))

    # ========================================================
    # 2. Spearman rank correlation
    # scipy を使わず pandas で計算
    # ========================================================

    spearman_rho = pd.Series(logf).corr(
        pd.Series(logP),
        method="spearman"
    )

    # ========================================================
    # 3. adjacent monotonicity
    # ========================================================

    dlogP = np.diff(logP)
    dlogf = np.diff(logf)

    adjacent_decrease_fraction = np.mean(dlogP < 0)

    # local alpha = - dlogP / dlogf
    local_alpha = -dlogP / dlogf

    positive_local_slope_fraction = np.mean(local_alpha > 0)
    median_local_alpha = np.nanmedian(local_alpha)

    # 低周波端から高周波端まで何 dex 落ちたか
    log10_drop_low_to_high = logP[0] - logP[-1]

    # 完全単調減少なら 1、増加が多いほど小さくなる score
    decreasing_variation = np.sum(np.maximum(-dlogP, 0))
    total_variation = np.sum(np.abs(dlogP))

    if total_variation > 0:
        monotonic_decrease_score = decreasing_variation / total_variation
    else:
        monotonic_decrease_score = np.nan

    return pd.Series({
        "n_points": len(f),
        "alpha": alpha,
        "intercept": intercept,
        "r2_loglog": r2,
        "rmse_log10": rmse_log10,
        "spearman_rho": spearman_rho,
        "adjacent_decrease_fraction": adjacent_decrease_fraction,
        "positive_local_slope_fraction": positive_local_slope_fraction,
        "median_local_alpha": median_local_alpha,
        "log10_drop_low_to_high": log10_drop_low_to_high,
        "monotonic_decrease_score": monotonic_decrease_score,
    })

In [ ]:
import numpy as np
import pandas as pd


def make_monotonic_selected_times(
    B64_spectra_qf,
    selected_times,
    selected_fracs=None,
    freq_dim="spec_bins",
    fmin=3,
    fmax=32,
    alpha_min=0.0,
    alpha_max=None,
    r2_min=0.5,
    spearman_max=-0.7,
    adjacent_decrease_fraction_min=0.65,
    monotonic_decrease_score_min=0.7,
):
    """
    selected_times のうち、B PSD が概ね単調減少する時刻だけを抽出する。

    Returns
    -------
    selected_times_mono : np.ndarray
        条件を満たす更新版 selected_times
    selected_fracs_mono : np.ndarray or None
        selected_fracs が与えられていれば同じ mask で抽出したもの
    metrics_df : pd.DataFrame
        各 selected_time の monotonicity metrics
    mask_mono : np.ndarray
        selected_times に対応する bool mask
    """

    rows = []

    selected_times_pd = pd.DatetimeIndex(pd.to_datetime(selected_times))

    for i, t in enumerate(selected_times_pd):
        try:
            psd_i = B64_spectra_qf.sel(time=t)
        except KeyError:
            # 念のため。B64_spectra_qf が selected_times と完全一致していない場合は NaN 扱い。
            row = {
                "index": i,
                "time": t,
                "n_points": np.nan,
                "alpha": np.nan,
                "intercept": np.nan,
                "r2_loglog": np.nan,
                "rmse_log10": np.nan,
                "spearman_rho": np.nan,
                "adjacent_decrease_fraction": np.nan,
                "positive_local_slope_fraction": np.nan,
                "median_local_alpha": np.nan,
                "log10_drop_low_to_high": np.nan,
                "monotonic_decrease_score": np.nan,
                "is_monotonic_decreasing": False,
            }
            rows.append(row)
            continue

        metrics = quantify_decreasing_psd(
            psd_i,
            freq_dim=freq_dim,
            fmin=fmin,
            fmax=fmax,
        )

        alpha = metrics["alpha"]
        r2 = metrics["r2_loglog"]
        rho = metrics["spearman_rho"]
        dec_frac = metrics["adjacent_decrease_fraction"]
        mono_score = metrics["monotonic_decrease_score"]

        ok = (
            np.isfinite(alpha)
            and np.isfinite(r2)
            and np.isfinite(rho)
            and np.isfinite(dec_frac)
            and np.isfinite(mono_score)
            and (alpha > alpha_min)
            and (r2 > r2_min)
            and (rho < spearman_max)
            and (dec_frac > adjacent_decrease_fraction_min)
            and (mono_score > monotonic_decrease_score_min)
        )

        if alpha_max is not None:
            ok = ok and (alpha < alpha_max)

        row = metrics.to_dict()
        row["index"] = i
        row["time"] = t
        row["is_monotonic_decreasing"] = bool(ok)

        rows.append(row)

    metrics_df = pd.DataFrame(rows)
    metrics_df = metrics_df.set_index("time")

    mask_mono = metrics_df["is_monotonic_decreasing"].values.astype(bool)

    selected_times_mono = np.asarray(selected_times)[mask_mono]

    if selected_fracs is not None:
        selected_fracs_mono = np.asarray(selected_fracs)[mask_mono]
    else:
        selected_fracs_mono = None

    return selected_times_mono, selected_fracs_mono, metrics_df, mask_mono

In [ ]:
import numpy as np
import pandas as pd
from tqdm.contrib.concurrent import thread_map


def quantify_decreasing_psd_array(f, P):
    """
    1D PSD array に対して、概ね単調減少かを定量化する。
    xarray を使わない高速版。
    """

    f = np.asarray(f, dtype=float)
    P = np.asarray(P, dtype=float)

    use = np.isfinite(f) & np.isfinite(P) & (P > 0)

    f = f[use]
    P = P[use]

    if len(f) < 3:
        return {
            "n_points": len(f),
            "alpha": np.nan,
            "intercept": np.nan,
            "r2_loglog": np.nan,
            "rmse_log10": np.nan,
            "spearman_rho": np.nan,
            "adjacent_decrease_fraction": np.nan,
            "positive_local_slope_fraction": np.nan,
            "median_local_alpha": np.nan,
            "log10_drop_low_to_high": np.nan,
            "monotonic_decrease_score": np.nan,
        }

    order = np.argsort(f)
    f = f[order]
    P = P[order]

    logf = np.log10(f)
    logP = np.log10(P)

    # logP = intercept + slope * logf
    slope, intercept = np.polyfit(logf, logP, 1)
    alpha = -slope

    fit = intercept + slope * logf
    resid = logP - fit

    ss_res = np.sum(resid**2)
    ss_tot = np.sum((logP - np.mean(logP))**2)

    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse_log10 = np.sqrt(np.mean(resid**2))

    spearman_rho = pd.Series(logf).corr(
        pd.Series(logP),
        method="spearman",
    )

    dlogP = np.diff(logP)
    dlogf = np.diff(logf)

    adjacent_decrease_fraction = np.mean(dlogP < 0)

    local_alpha = -dlogP / dlogf

    positive_local_slope_fraction = np.mean(local_alpha > 0)
    median_local_alpha = np.nanmedian(local_alpha)

    log10_drop_low_to_high = logP[0] - logP[-1]

    decreasing_variation = np.sum(np.maximum(-dlogP, 0))
    total_variation = np.sum(np.abs(dlogP))

    if total_variation > 0:
        monotonic_decrease_score = decreasing_variation / total_variation
    else:
        monotonic_decrease_score = np.nan

    return {
        "n_points": len(f),
        "alpha": alpha,
        "intercept": intercept,
        "r2_loglog": r2,
        "rmse_log10": rmse_log10,
        "spearman_rho": spearman_rho,
        "adjacent_decrease_fraction": adjacent_decrease_fraction,
        "positive_local_slope_fraction": positive_local_slope_fraction,
        "median_local_alpha": median_local_alpha,
        "log10_drop_low_to_high": log10_drop_low_to_high,
        "monotonic_decrease_score": monotonic_decrease_score,
    }


def _check_threshold(value, threshold, mode):
    """
    threshold が None なら条件を課さない。
    mode は 'gt' or 'lt'。
    """

    if threshold is None:
        return True

    if not np.isfinite(value):
        return False

    if mode == "gt":
        return value > threshold
    elif mode == "lt":
        return value < threshold
    else:
        raise ValueError("mode must be 'gt' or 'lt'")

In [ ]:
from tqdm.contrib.concurrent import thread_map

def make_monotonic_selected_times_parallel(
    B64_spectra_qf,
    selected_times,
    selected_fracs=None,
    freq_dim="spec_bins",
    fmin=3,
    fmax=32,
    alpha_min=0.0,
    alpha_max=None,
    r2_min=0.5,
    spearman_max=-0.7,
    adjacent_decrease_fraction_min=0.65,
    monotonic_decrease_score_min=0.7,
    n_workers=8,
    desc="Checking monotonic B PSD",
    nearest_tolerance=None,
):
    """
    selected_times のうち、B PSD が概ね単調減少する時刻だけを抽出する。
    並列化 + tqdm 版。

    threshold に None を指定した場合、その条件は使わない。
    """

    da = B64_spectra_qf.transpose("time", freq_dim)
    da = da.sel({freq_dim: slice(fmin, fmax)})

    all_times = pd.DatetimeIndex(pd.to_datetime(da.time.values))
    selected_times_pd = pd.DatetimeIndex(pd.to_datetime(selected_times))

    freqs = da[freq_dim].values.astype(float)
    spectra_values = da.values.astype(float)

    # selected_times -> B64_spectra_qf の time index
    if nearest_tolerance is None:
        time_indexer = all_times.get_indexer(selected_times_pd)
    else:
        time_indexer = all_times.get_indexer(
            selected_times_pd,
            method="nearest",
            tolerance=pd.Timedelta(nearest_tolerance),
        )

    def process_one(i):
        t = selected_times_pd[i]
        idx = time_indexer[i]

        if idx < 0:
            return {
                "index": i,
                "time": t,
                "n_points": np.nan,
                "alpha": np.nan,
                "intercept": np.nan,
                "r2_loglog": np.nan,
                "rmse_log10": np.nan,
                "spearman_rho": np.nan,
                "adjacent_decrease_fraction": np.nan,
                "positive_local_slope_fraction": np.nan,
                "median_local_alpha": np.nan,
                "log10_drop_low_to_high": np.nan,
                "monotonic_decrease_score": np.nan,
                "is_monotonic_decreasing": False,
            }

        P = spectra_values[idx, :]

        metrics = quantify_decreasing_psd_array(
            f=freqs,
            P=P,
        )

        alpha = metrics["alpha"]
        r2 = metrics["r2_loglog"]
        rho = metrics["spearman_rho"]
        dec_frac = metrics["adjacent_decrease_fraction"]
        mono_score = metrics["monotonic_decrease_score"]

        ok = (
            np.isfinite(alpha)
            and np.isfinite(r2)
            and np.isfinite(rho)
            and np.isfinite(dec_frac)
            and np.isfinite(mono_score)
            and _check_threshold(alpha, alpha_min, "gt")
            and _check_threshold(alpha, alpha_max, "lt")
            and _check_threshold(r2, r2_min, "gt")
            and _check_threshold(rho, spearman_max, "lt")
            and _check_threshold(dec_frac, adjacent_decrease_fraction_min, "gt")
            and _check_threshold(mono_score, monotonic_decrease_score_min, "gt")
        )

        row = dict(metrics)
        row["index"] = i
        row["time"] = t
        row["matched_time"] = all_times[idx]
        row["is_monotonic_decreasing"] = bool(ok)

        return row

    rows = thread_map(
        process_one,
        range(len(selected_times_pd)),
        max_workers=n_workers,
        desc=desc,
    )

    metrics_df = pd.DataFrame(rows)
    metrics_df = metrics_df.set_index("time")

    mask_mono = metrics_df["is_monotonic_decreasing"].values.astype(bool)

    selected_times_mono = np.asarray(selected_times)[mask_mono]

    if selected_fracs is not None:
        selected_fracs_mono = np.asarray(selected_fracs)[mask_mono]
    else:
        selected_fracs_mono = None

    return selected_times_mono, selected_fracs_mono, metrics_df, mask_mono

In [ ]:
selected_times_mono, selected_fracs_mono, B64_mono_metrics_df, mask_mono = make_monotonic_selected_times_parallel(
    B64_spectra_qf=B64_spectra_qf,
    selected_times=selected_times,
    selected_fracs=selected_fracs,
    freq_dim="spec_bins",
    fmin=3,
    fmax=32,

    alpha_min=0,
    alpha_max=None,
    r2_min=0.5,
    spearman_max=-0.7,
    adjacent_decrease_fraction_min=None,
    monotonic_decrease_score_min=0.5,

    n_workers=12,
)

print(f"元の selected_times 数: {len(selected_times)}")
print(f"B PSD が概ね単調減少する selected_times 数: {len(selected_times_mono)}")
print(f"残存率: {len(selected_times_mono) / len(selected_times):.3f}")

# Bzの角度の設定

Breneman+ 2022のRBSPの設定に従い、磁場とspin planeの角度が15°以上の時に有効とする。

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def make_EdotB0_geometry_mask(
    B_da,
    angle_min_deg=15.0,
    time_dim="time",
    comp_dim=None,
):
    """
    B_da: xarray.DataArray
        shape = (time, 3) を想定。
        成分順は [Bx, By, Bz]。

    angle_min_deg:
        磁場が spin plane / DSI xy plane から何度以上離れていれば有効とみなすか。
        RBSP 的には 15 deg が一つの目安。

    returns:
        xr.Dataset containing:
            B_spinplane_angle_deg
            Ez_amp_factor
            bad_EdotB0_geometry
    """

    if comp_dim is None:
        comp_dims = [d for d in B_da.dims if d != time_dim]
        if len(comp_dims) != 1:
            raise ValueError("component dimension を特定できないので comp_dim を指定してほしい。")
        comp_dim = comp_dims[0]

    Bx = B_da.isel({comp_dim: 0})
    By = B_da.isel({comp_dim: 1})
    Bz = B_da.isel({comp_dim: 2})

    Bxy = np.sqrt(Bx**2 + By**2)
    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)

    theta_B_spin = np.degrees(np.arctan2(np.abs(Bz), Bxy))

    Ez_amp_factor = xr.where(
        np.abs(Bz) > 0,
        Bxy / np.abs(Bz),
        np.inf,
    )

    bad_geom = (
        ~np.isfinite(theta_B_spin)
        | ~np.isfinite(Ez_amp_factor)
        | ~np.isfinite(Bmag)
        | (Bmag <= 0)
        | (theta_B_spin < angle_min_deg)
    )

    out = xr.Dataset({
        "B_spinplane_angle_deg": theta_B_spin,
        "Ez_amp_factor": Ez_amp_factor,
        "bad_EdotB0_geometry": bad_geom,
    })

    return out

In [ ]:
geom = make_EdotB0_geometry_mask(
    B64_data_dsi_qf,
    angle_min_deg=15.0,
)

bad_mask_spin_angle = geom["bad_EdotB0_geometry"]

bad_times_spin_angle = B64_data_dsi_qf["time"].where(bad_mask_spin_angle, drop=True)

print(bad_times_spin_angle)
print("bad fraction =", float(bad_mask_spin_angle.mean()))

# EFD波形データの有効時間の確認

In [ ]:
ergpy.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', get_support_data=True)
E64_data_dsi_quality_flag = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_quality_flag', xarray=True).sortby('time')

# 第二段階検証

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def keep_true_runs_1d(
    cond,
    dim="time",
    min_run_length=2,
    check_time_gap=True,
    gap_factor=1.5,
):
    """
    1D boolean DataArray について、True が min_run_length 点以上
    連続する区間だけ True として残す。

    Parameters
    ----------
    cond : xr.DataArray
        1D boolean DataArray
    dim : str
        time dimension name
    min_run_length : int
        何点以上連続した True を残すか
    check_time_gap : bool
        True の場合、時刻 gap が通常 dt から大きく外れると別 run とみなす
    gap_factor : float
        median dt の gap_factor 倍より大きい gap で run を切る

    Returns
    -------
    keep_da : xr.DataArray
        cond と同じ time 軸を持つ boolean DataArray
    """

    cond = cond.fillna(False).astype(bool)
    values = cond.values.astype(bool)

    n = len(values)
    keep = np.zeros(n, dtype=bool)

    if n == 0:
        return xr.DataArray(
            keep,
            coords={dim: cond[dim]},
            dims=(dim,),
            name="consecutive_true_mask",
        )

    times = pd.DatetimeIndex(pd.to_datetime(cond[dim].values))

    if check_time_gap and n >= 2:
        t_ns = times.asi8
        dt_sec = np.diff(t_ns) * 1e-9
        dt_sec_valid = dt_sec[np.isfinite(dt_sec) & (dt_sec > 0)]

        if len(dt_sec_valid) > 0:
            dt_ref = np.median(dt_sec_valid)
            max_gap_sec = gap_factor * dt_ref
        else:
            max_gap_sec = np.inf
    else:
        max_gap_sec = np.inf

    i = 0

    while i < n:
        if not values[i]:
            i += 1
            continue

        j = i + 1

        while j < n and values[j]:
            if check_time_gap:
                gap_sec = (times[j] - times[j - 1]).total_seconds()
                if gap_sec > max_gap_sec:
                    break

            j += 1

        run_length = j - i

        if run_length >= min_run_length:
            keep[i:j] = True

        i = j

    keep_da = xr.DataArray(
        keep,
        coords={dim: cond[dim]},
        dims=(dim,),
        name="consecutive_true_mask",
    )

    return keep_da

In [ ]:
bad_mask_raw = qf_B > 21

bad_mask_mgf = keep_true_runs_1d(
    bad_mask_raw,
    dim="time",
    min_run_length=4,     # 10点以上連続した bad のみ残す
    check_time_gap=True,
    gap_factor=1.5,
)

bad_times_mgf = qf_B.time.where(bad_mask_mgf, drop=True)

print(f"raw bad times: {int(bad_mask_raw.sum().values)}")
print(f"consecutive bad times: {int(bad_mask_mgf.sum().values)}")
print(bad_times_mgf)

In [ ]:
bad_mask_efd_wave_raw = E64_data_dsi_quality_flag != 0

bad_mask_efd_wave = keep_true_runs_1d(
    bad_mask_efd_wave_raw,
    dim='time',
    min_run_length=10,
    check_time_gap=True,
    gap_factor=1.5,
)

bad_times_efd_wave = E64_data_dsi_quality_flag.time.where(bad_mask_efd_wave, drop=True)

print(f"raw bad times: {int(bad_mask_efd_wave_raw.sum().values)}")
print(f"consecutive bad times: {int(bad_mask_efd_wave.sum().values)}")
print(bad_times_efd_wave)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def find_time_gap_ranges(
    time_da,
    threshold=pd.Timedelta(seconds=30),
    time_dim="time",
    reason="time_gap",
):
    """
    time 軸で threshold 以上の間隔が空いた箇所を bad gap として抽出する。

    Parameters
    ----------
    time_da : xr.DataArray
        time 座標。例: qf_B.time
    threshold : pd.Timedelta
        隣接時刻差がこの値以上なら gap とみなす。
    time_dim : str
        time dimension name.
    reason : str
        gap の reason 名。

    Returns
    -------
    gap_ranges_df : pd.DataFrame
        gap_start_time, gap_end_time, gap_duration_sec などを持つ表。
    """

    times = pd.DatetimeIndex(pd.to_datetime(time_da.values)).sort_values()

    columns = [
        "gap_start_time",
        "gap_end_time",
        "gap_duration_sec",
        "gap_duration_min",
        "reason",
    ]

    if len(times) < 2:
        return pd.DataFrame(columns=columns)

    dt = times[1:] - times[:-1]
    gap_mask = dt >= threshold

    rows = []

    for i in np.where(gap_mask)[0]:
        gap_duration_sec = dt[i].total_seconds()

        rows.append({
            "gap_start_time": times[i],
            "gap_end_time": times[i + 1],
            "gap_duration_sec": gap_duration_sec,
            "gap_duration_min": gap_duration_sec / 60,
            "reason": reason,
        })

    gap_ranges_df = pd.DataFrame(rows, columns=columns)

    if len(gap_ranges_df) == 0:
        return gap_ranges_df

    gap_ranges_df = gap_ranges_df.sort_values(
        ["gap_start_time", "gap_end_time"]
    ).reset_index(drop=True)

    return gap_ranges_df

In [ ]:
mgf_gap_ranges_df = find_time_gap_ranges(
    B64_data_dsi_qf.time,
    threshold=pd.Timedelta(milliseconds=63),
    reason="mgf_time_gap",
)

print(mgf_gap_ranges_df)

In [ ]:
efd_wave_gap_ranges_df = find_time_gap_ranges(
    E64_data_dsi_quality_flag.time,
    threshold=pd.Timedelta(seconds=30),
    reason="mgf_time_gap",
)

print(efd_wave_gap_ranges_df)

In [ ]:
import pandas as pd
import numpy as np


# ============================================================
# helper functions
# ============================================================

def _as_datetime_index_from_time_da(time_da):
    """
    xarray DataArray / numpy array / list-like を DatetimeIndex に変換する。
    """
    if time_da is None:
        return pd.DatetimeIndex([])

    if hasattr(time_da, "values"):
        vals = time_da.values
    else:
        vals = time_da

    out = pd.DatetimeIndex(pd.to_datetime(vals))
    out = out.dropna().sort_values().unique()
    out = pd.DatetimeIndex(out)

    return out


def point_times_to_intervals(
    times,
    gap_factor=1.5,
):
    """
    bad_times のような点列を、連続点ごとに interval 化する。

    例:
      t0, t0+1/64s, t0+2/64s, ... を
      [t0, tN] の1区間にまとめる。

    単発 bad time は [t, t] として残る。
    """

    times = _as_datetime_index_from_time_da(times)

    if len(times) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    if len(times) == 1:
        return pd.DataFrame({
            "start_time": times,
            "end_time": times,
        })

    t_ns = times.asi8
    dt_sec = np.diff(t_ns) * 1e-9

    dt_valid = dt_sec[np.isfinite(dt_sec) & (dt_sec > 0)]

    if len(dt_valid) == 0:
        return pd.DataFrame({
            "start_time": times,
            "end_time": times,
        })

    dt_ref = np.median(dt_valid)
    max_gap = gap_factor * dt_ref

    is_continuous = dt_sec <= max_gap

    break_points = np.where(~is_continuous)[0] + 1
    idx_groups = np.split(np.arange(len(times)), break_points)

    intervals = []

    for idx in idx_groups:
        intervals.append((times[idx[0]], times[idx[-1]]))

    return pd.DataFrame(intervals, columns=["start_time", "end_time"])


def gap_df_to_intervals(gap_df):
    """
    LEPe_gap_ranges_df / LEPi_gap_ranges_df を
    start_time, end_time の DataFrame に変換する。

    対応列:
      - missing_start_time, missing_end_time
      - gap_start_time, gap_end_time
    """

    if gap_df is None or len(gap_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    if {"missing_start_time", "missing_end_time"}.issubset(gap_df.columns):
        start_col = "missing_start_time"
        end_col   = "missing_end_time"
    elif {"gap_start_time", "gap_end_time"}.issubset(gap_df.columns):
        start_col = "gap_start_time"
        end_col   = "gap_end_time"
    else:
        raise ValueError(
            "gap_df must have either "
            "['missing_start_time', 'missing_end_time'] or "
            "['gap_start_time', 'gap_end_time'] columns."
        )

    out = pd.DataFrame({
        "start_time": pd.to_datetime(gap_df[start_col]),
        "end_time": pd.to_datetime(gap_df[end_col]),
    })

    out = out.dropna()
    out = out[out["end_time"] >= out["start_time"]]

    return out.reset_index(drop=True)


def merge_intervals(intervals_df):
    """
    start_time, end_time を持つ interval DataFrame を merge する。
    """

    if intervals_df is None or len(intervals_df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    df = intervals_df.dropna().copy()
    df = df[df["end_time"] >= df["start_time"]]
    df = df.sort_values("start_time").reset_index(drop=True)

    if len(df) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    starts = pd.DatetimeIndex(pd.to_datetime(df["start_time"])).to_numpy()
    ends   = pd.DatetimeIndex(pd.to_datetime(df["end_time"])).to_numpy()

    merged = []

    cur_start = starts[0]
    cur_end = ends[0]

    for s, e in zip(starts[1:], ends[1:]):
        if s <= cur_end:
            cur_end = max(cur_end, e)
        else:
            merged.append((cur_start, cur_end))
            cur_start = s
            cur_end = e

    merged.append((cur_start, cur_end))

    return pd.DataFrame({
        "start_time": pd.to_datetime([m[0] for m in merged]),
        "end_time": pd.to_datetime([m[1] for m in merged]),
    })


def make_forbidden_intervals_for_mono(
    bad_times,
    bad_times_mgf,
    bad_times_spin_angle,
    bad_times_efd_wave,
    LEPe_gap_ranges_df=None,
    LEPi_gap_ranges_df=None,
    mgf_gap_ranges_df=None,
    efd_wave_gap_ranges_df=None,
    att_gap_ranges_df=None,
):
    """
    EFD bad times, MGF bad times, LEPe gap, LEPi gap を
    まとめて禁止 interval にする。
    """

    dfs = []

    bad_efd_intervals = point_times_to_intervals(bad_times)
    bad_mgf_intervals = point_times_to_intervals(bad_times_mgf)
    bad_spin_angle_intervals = point_times_to_intervals(bad_times_spin_angle)
    bad_efd_wave_intervals = point_times_to_intervals(bad_times_efd_wave)

    if len(bad_efd_intervals) > 0:
        dfs.append(bad_efd_intervals)

    if len(bad_mgf_intervals) > 0:
        dfs.append(bad_mgf_intervals)

    if len(bad_spin_angle_intervals) > 0:
        dfs.append(bad_spin_angle_intervals)

    if len(bad_efd_wave_intervals) > 0:
        dfs.append(bad_efd_wave_intervals)

    lepe_intervals = gap_df_to_intervals(LEPe_gap_ranges_df)
    lepi_intervals = gap_df_to_intervals(LEPi_gap_ranges_df)
    mgf_intervals = gap_df_to_intervals(mgf_gap_ranges_df)
    efd_intervals = gap_df_to_intervals(efd_wave_gap_ranges_df)
    att_intervals = gap_df_to_intervals(att_gap_ranges_df)

    if len(lepe_intervals) > 0:
        dfs.append(lepe_intervals)

    if len(lepi_intervals) > 0:
        dfs.append(lepi_intervals)

    if len(mgf_intervals) > 0:
        dfs.append(mgf_intervals)

    if len(efd_intervals) > 0:
        dfs.append(efd_intervals)

    if len(att_intervals) > 0:
        dfs.append(att_intervals)

    if len(dfs) == 0:
        return pd.DataFrame(columns=["start_time", "end_time"])

    forbidden_intervals_df = pd.concat(dfs, ignore_index=True)
    #forbidden_intervals_df = merge_intervals(forbidden_intervals_df)

    return forbidden_intervals_df

In [ ]:
def make_trimmed_time_ranges_by_forbidden(
    selected_times,
    forbidden_intervals_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
):
    """
    selected_times ± half_width の時間範囲を作る。
    ただし forbidden intervals を含まないように、
    各 selected time の前後で最も近い forbidden interval まで切り詰める。
    """

    selected_times_pd = pd.DatetimeIndex(
        pd.to_datetime(selected_times)
    ).sort_values()

    if len(selected_times_pd) == 0:
        return []

    start0 = selected_times_pd - half_width
    end0   = selected_times_pd + half_width

    if forbidden_intervals_df is None or len(forbidden_intervals_df) == 0:
        return list(zip(start0, end0))

    forbidden_intervals_df = forbidden_intervals_df.sort_values("start_time").reset_index(drop=True)

    f_starts = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["start_time"]))
    f_ends   = pd.DatetimeIndex(pd.to_datetime(forbidden_intervals_df["end_time"]))

    t_np      = selected_times_pd.to_numpy()
    start_np  = start0.to_numpy()
    end_np    = end0.to_numpy()
    fs_np     = f_starts.to_numpy()
    fe_np     = f_ends.to_numpy()

    # 各 selected time に対して、start <= t となる最後の forbidden interval
    i_prev = np.searchsorted(fs_np, t_np, side="right") - 1
    valid_prev = i_prev >= 0

    inside_forbidden = np.zeros(len(t_np), dtype=bool)

    # selected time 自体が forbidden interval 内なら除外
    inside_forbidden[valid_prev] = (
        t_np[valid_prev] <= fe_np[i_prev[valid_prev]]
    )

    # 前側の forbidden interval で start を切る
    start_trim = start_np.copy()

    use_prev_trim = np.zeros(len(t_np), dtype=bool)
    use_prev_trim[valid_prev] = (
        fe_np[i_prev[valid_prev]] > start_np[valid_prev]
    )

    start_trim[use_prev_trim] = (
        fe_np[i_prev[use_prev_trim]] + np.timedelta64(eps.value, "ns")
    )

    # 後側の forbidden interval で end を切る
    i_next = i_prev + 1
    valid_next = i_next < len(fs_np)

    end_trim = end_np.copy()

    use_next_trim = np.zeros(len(t_np), dtype=bool)
    use_next_trim[valid_next] = (
        fs_np[i_next[valid_next]] < end_np[valid_next]
    )

    end_trim[use_next_trim] = (
        fs_np[i_next[use_next_trim]] - np.timedelta64(eps.value, "ns")
    )

    valid = (~inside_forbidden) & (end_trim > start_trim)

    start_out = pd.to_datetime(start_trim[valid])
    end_out   = pd.to_datetime(end_trim[valid])

    return list(zip(start_out, end_out))


def merge_time_ranges(
    time_ranges,
    max_duration=None,
):
    """
    overlapping time ranges を merge する。
    max_duration を指定した場合は、それを超える merge はしない。
    """

    if len(time_ranges) == 0:
        return []

    time_ranges = sorted(time_ranges)

    merged = []

    for start, end in time_ranges:
        if not merged:
            merged.append((start, end))
            continue

        prev_start, prev_end = merged[-1]

        is_overlapping = start <= prev_end

        candidate_start = prev_start
        candidate_end = max(prev_end, end)

        if max_duration is None:
            ok_duration = True
        else:
            ok_duration = (candidate_end - candidate_start) <= max_duration

        if is_overlapping and ok_duration:
            merged[-1] = (candidate_start, candidate_end)
        else:
            merged.append((start, end))

    return merged

In [ ]:
point_times_to_intervals(bad_times_spin_angle)

In [ ]:
import time

t0_clock = time.perf_counter()

# ============================================================
# forbidden intervals:
# EFD bad + MGF bad + LEPe gap + LEPi gap
# ============================================================

forbidden_intervals_mono_df = make_forbidden_intervals_for_mono(
    bad_times=bad_times,
    bad_times_mgf=bad_times_mgf,
    bad_times_spin_angle=bad_times_spin_angle,
    bad_times_efd_wave=bad_times_efd_wave,
    LEPe_gap_ranges_df=LEPe_omniflux_gap_ranges_df,
    LEPi_gap_ranges_df=LEPi_omniflux_gap_ranges_df,
    mgf_gap_ranges_df=mgf_gap_ranges_df,
    efd_wave_gap_ranges_df=efd_wave_gap_ranges_df,
    att_gap_ranges_df=att_nan_intervals_df
)

print(f"forbidden intervals: {len(forbidden_intervals_mono_df)}")

forbidden_intervals_df

In [ ]:
# 確認用
bad_times_efd_pd = _as_datetime_index_from_time_da(bad_times)
bad_times_mgf_pd = _as_datetime_index_from_time_da(bad_times_mgf)
bad_times_spin_angle_pd = _as_datetime_index_from_time_da(bad_times_spin_angle)

print(f"EFD bad_times: {len(bad_times_efd_pd)}")
print(f"MGF bad_times: {len(bad_times_mgf_pd)}")
print(f"Spin angle bad_times: {len(bad_times_spin_angle_pd)}")
print(f"LEPe gaps: {len(LEPe_omniflux_gap_ranges_df) if LEPe_omniflux_gap_ranges_df is not None else 0}")
print(f"LEPi gaps: {len(LEPi_omniflux_gap_ranges_df) if LEPi_omniflux_gap_ranges_df is not None else 0}")


# ============================================================
# selected_times_mono ±15 min を forbidden intervals で切る
# ============================================================

time_ranges_mono = make_trimmed_time_ranges_by_forbidden(
    selected_times=selected_times_mono,
    forbidden_intervals_df=forbidden_intervals_mono_df,
    half_width=pd.Timedelta(minutes=15),
    eps=pd.Timedelta(nanoseconds=1),
)

print(f"time ranges before merge: {len(time_ranges_mono)}")


# ============================================================
# Merge overlapping time ranges
# ============================================================

merged_ranges_mono = merge_time_ranges(
    time_ranges_mono,
    max_duration=None,
)


# ============================================================
# DataFrame 化
# ============================================================

merged_ranges_mono_df = pd.DataFrame(
    merged_ranges_mono,
    columns=["start_time", "end_time"]
)

if len(merged_ranges_mono_df) > 0:
    merged_ranges_mono_df["duration_minutes"] = (
        merged_ranges_mono_df["end_time"] - merged_ranges_mono_df["start_time"]
    ).dt.total_seconds() / 60

    # 30分を超える時間範囲のみ有効にする
    merged_ranges_mono_df = merged_ranges_mono_df[
        merged_ranges_mono_df["duration_minutes"] > 30
    ].reset_index(drop=True)
else:
    merged_ranges_mono_df["duration_minutes"] = []

elapsed = time.perf_counter() - t0_clock

print(f"元の選択時間数: {len(selected_times_mono)}")
print(f"30分超のマージ後時間範囲数: {len(merged_ranges_mono_df)}")
print(f"elapsed: {elapsed:.2f} sec")
print("\n有効なマージ時間範囲:")
print(merged_ranges_mono_df)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
path_base_save_plot = Path(path_base_save_plot + f'/PWE-EFD_spec_after_B')
os.makedirs(path_base_save_plot, exist_ok=True)

def edges_numeric(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return np.concatenate((arr, arr + 1.0))
    d = np.diff(arr) / 2.0
    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))

vmax = np.log10(1E1)
vmin = np.log10(1e-4)

for idx, row in merged_ranges_mono_df.iterrows():
    start = row["start_time"]
    end = row["end_time"]

    if pd.isna(start) or pd.isna(end):
        continue

    subset_range = efd_spectra_qf.sel(time=slice(start, end))

    if subset_range.sizes.get("time", 0) < 2:
        continue

    times = pd.to_datetime(subset_range.time.values)

    if "spec_bins" in subset_range.coords:
        freqs = subset_range["spec_bins"].values
    elif "v" in subset_range.coords:
        freqs = subset_range["v"].values
    else:
        freqs = np.arange(subset_range.sizes["v_dim"])

    data_range = subset_range.values
    data_range = np.where(
        np.isfinite(data_range) & (data_range > 0),
        data_range,
        np.nan
    )

    logdata_range = np.log10(data_range)

    if np.isnan(logdata_range).all():
        continue

    t_num = mdates.date2num(times.to_pydatetime())
    t_edges = edges_numeric(t_num)
    f_edges = edges_numeric(freqs)

    fig_i, ax_i = plt.subplots(figsize=(12, 4.5))

    pcm_i = ax_i.pcolormesh(
        t_edges,
        f_edges,
        logdata_range.T,
        shading="auto",
        cmap="turbo",
        norm=Normalize(vmin=vmin, vmax=vmax),
    )

    f_cH_plot = f_cH.sel(time=slice(start, end))
    f_cHe_plot = f_cHe.sel(time=slice(start, end))
    f_cO_plot = f_cO.sel(time=slice(start, end))

    ax_i.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
    ax_i.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
    ax_i.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')

    ax_i.set_xlabel("Time")
    ax_i.xaxis_date()
    ax_i.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax_i.set_ylabel("Frequency [Hz]")
    ax_i.set_title(
        f"PWE-EFD spec ({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
    )
    ax_i.set_yscale("log")
    ax_i.set_ylim(3, 32)
    ax_i.set_xlim(start.to_pydatetime(), end.to_pydatetime())

    fig_i.colorbar(pcm_i, ax=ax_i, label=r"log10 PSD [$\mathrm{(mV/m)}^2/\mathrm{Hz}$]")
    plt.tight_layout()

    # =========================
    # save figure
    # =========================

    # start の月・日で階層フォルダを作る
    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"efd_spectra_qf_"
        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
    )

    save_path = save_dir / filename

    fig_i.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig_i)

    print(f"saved: {save_path}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize
from pathlib import Path


def edges_numeric(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 2:
        return np.concatenate((arr, arr + 1.0))
    d = np.diff(arr) / 2.0
    return np.concatenate(([arr[0] - d[0]], arr[:-1] + d, [arr[-1] + d[-1]]))


# ============================================================
# save base path
# ============================================================

path_base_save_plot = f'/mnt/j/statistical_analysis_arase/preanalysis'
path_base_save_plot = Path(path_base_save_plot + f'/MGF_spec_after_B')
os.makedirs(path_base_save_plot, exist_ok=True)



# ============================================================
# plot settings
# ============================================================

vmin = -4
vmax = -1

for idx, row in merged_ranges_mono_df.iterrows():

    start = pd.to_datetime(row["start_time"])
    end   = pd.to_datetime(row["end_time"])

    if pd.isna(start) or pd.isna(end):
        continue

    f_cH_plot = f_cH.sel(time=slice(start, end))
    f_cHe_plot = f_cHe.sel(time=slice(start, end))
    f_cO_plot = f_cO.sel(time=slice(start, end))

    # xarray sel 用
    B64_spectra_qf_analysis = B64_spectra_qf.sel(
        time=slice(start, end)
    )

    # time が少なすぎる場合は skip
    if B64_spectra_qf_analysis.sizes.get("time", 0) < 2:
        print(f"skip: too few time points: {start} - {end}")
        continue

    data = B64_spectra_qf_analysis.values
    data = np.where(np.isfinite(data) & (data > 0), data, np.nan)

    if np.isnan(data).all():
        print(f"skip: all NaN: {start} - {end}")
        continue

    logdata = np.log10(data)

    times = pd.DatetimeIndex(pd.to_datetime(B64_spectra_qf_analysis.time.values))
    t_num = mdates.date2num(times.to_pydatetime())
    t_edges = edges_numeric(t_num)

    freqs = B64_spectra_qf_analysis.spec_bins.values
    f_edges = edges_numeric(freqs)

    fig, ax = plt.subplots(figsize=(12, 4.5))

    pcm = ax.pcolormesh(
        t_edges,
        f_edges,
        logdata.T,
        shading="auto",
        cmap="turbo",
        norm=Normalize(vmin=vmin, vmax=vmax),
    )

    ax.plot(f_cH_plot.time, f_cH_plot.data, lw=1, c='white', linestyle='solid')
    ax.plot(f_cHe_plot.time, f_cHe_plot.data, lw=1, c='magenta', linestyle='solid')
    ax.plot(f_cO_plot.time, f_cO_plot.data, lw=1, c='yellow', linestyle='solid')

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    ax.set_yscale("log")
    ax.set_ylim(3, 32)

    ax.set_xlim(start.to_pydatetime(), end.to_pydatetime())

    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency [Hz]")
    ax.set_title(
        f"MGF 64hz total spec "
        f"({start:%Y-%m-%d %H:%M:%S} - {end:%Y-%m-%d %H:%M:%S})"
    )

    fig.colorbar(
        pcm,
        ax=ax,
        label=r"log10 PSD [$\mathrm{nT}^2/\mathrm{Hz}$]"
    )

    plt.tight_layout()

    # ========================================================
    # save
    # ========================================================

    save_dir = path_base_save_plot / f"{start:%Y}" / f"{start:%m}"
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"MGF_B64_spectra_"
        f"{start:%Y%m%d_%H%M%S}_to_{end:%Y%m%d_%H%M%S}.png"
    )

    save_path = save_dir / filename

    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"saved: {save_path}")